# 1. Imports and Setup
This cell imports essential libraries for deep learning, tensor manipulation, image handling, and visualization:
- `torch` & `torch.nn`: Core PyTorch framework and neural network layers.
- `DataLoader`: Batching and dataset iteration.
- `datasets` & `transforms`: Official CIFAR-10 dataset and image transformations.
- `PIL.Image`: Loading and processing external RGB images.
- `matplotlib.pyplot`: Plotting and image display.
- `device`: Selects GPU (`cuda`) if available for accelerated execution, otherwise CPU.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Test Dataset Loading and Normalization
This cell configures test data evaluation:
- `transform_test`: Converts images to PyTorch tensors and normalizes 3 color channels (RGB) using CIFAR-10 channel-wise dataset means `(0.4914, 0.4822, 0.4465)` and standard deviations `(0.2023, 0.1994, 0.2010)`.
- `test_dataset`: Downloads/loads the 10,000 official CIFAR-10 test images.
- `test_loader`: Feeds test images in batches of size `256`.
- `CIFAR10_CLASSES`: Mapping list for the 10 class labels (`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`).

In [ ]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

test_dataset = datasets.CIFAR10(root='../../Datasets', train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# 3. Deep CNN Architecture Definition
This cell defines a 3-block 2D Convolutional Neural Network (`CIFAR10_CNN`) tailored for RGB images:
- **Block 1:** `Conv2d(3 -> 32)` -> `BatchNorm2d(32)` -> `ReLU` -> `MaxPool2d(2, 2)` (Reduces 32x32 to 16x16)
- **Block 2:** `Conv2d(32 -> 64)` -> `BatchNorm2d(64)` -> `ReLU` -> `MaxPool2d(2, 2)` (Reduces 16x16 to 8x8)
- **Block 3:** `Conv2d(64 -> 128)` -> `BatchNorm2d(128)` -> `ReLU` -> `MaxPool2d(2, 2)` (Reduces 8x8 to 4x4)
- **Classifier:** `Flatten()` -> `Dropout(0.5)` -> `LazyLinear(256)` -> `ReLU` -> `Linear(256 -> 10)`.
- `nn.BatchNorm2d`: Stabilizes activations across mini-batches, speeding up convergence.
- `nn.Dropout(0.5)`: Zeroes out 50% of neurons randomly during training to prevent overfitting.

In [ ]:
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.LazyLinear(256)
        self.relu4 = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu4(x)
        x = self.fc2(x)
        return x

# 4. Loading Pre-trained Weights
This cell initializes the model and loads trained weights directly:
- `dummy_input = torch.zeros(1, 3, 32, 32)`: Passes a dummy tensor through the network to initialize `nn.LazyLinear` dimensions (`128 * 4 * 4 = 2048`) prior to parameter restoration.
- `model.load_state_dict(...)`: Restores pre-trained weights from `cifar10_cnn_model.pth`.
- `model.eval()`: Freezes BatchNorm parameters and turns off Dropout for evaluation.

In [ ]:
MODEL_PATH = "cifar10_cnn_model.pth"

model = CIFAR10_CNN().to(device)
dummy_input = torch.zeros(1, 3, 32, 32).to(device)
model(dummy_input)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# 5. Evaluating Loaded Model Accuracy
This cell evaluates the loaded model's accuracy on the 10,000 CIFAR-10 test set without calculating gradients (`torch.no_grad()`).

In [ ]:
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

print(f"Loaded Model Test Accuracy: {100. * test_correct / test_total:.2f}%")

# 6. Batch Visualizations and Sample Predictions
This cell visualizes 10 test samples along with predictions:
- `denormalize(img)`: Reverses CIFAR-10 normalization (`img * std + mean`) to display true RGB colors.
- Plots a 2x5 grid showing images alongside `True` vs `Pred` labels (colored **green** if correct, **red** if incorrect).

In [ ]:
def denormalize(img):
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).reshape(3, 1, 1)
    std = torch.tensor([0.2023, 0.1994, 0.2010]).reshape(3, 1, 1)
    return img * std + mean

images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    _, preds = outputs.max(1)

plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    img = denormalize(images[i].cpu()).permute(1, 2, 0).numpy()
    img = img.clip(0, 1)
    plt.imshow(img)
    true_cls = CIFAR10_CLASSES[labels[i].item()]
    pred_cls = CIFAR10_CLASSES[preds[i].item()]
    color = "green" if true_cls == pred_cls else "red"
    plt.title(f"True: {true_cls}\nPred: {pred_cls}", color=color)
    plt.axis("off")

plt.tight_layout()
plt.show()

# 7. Single Image Inference Pipeline
This cell defines and executes a complete single-image inference pipeline:
- `predict_image(image_path)`: Loads any external image file, resizes it to `(32, 32)`, applies normalization, adds batch dimension (`unsqueeze(0)`), passes it to the network, and computes confidence scores via `torch.softmax`.
- Saves a test sample (`test_sample.jpg`) and runs `predict_image()` to demonstrate inference output.

In [ ]:
def predict_image(image_path):
    infer_transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])

    raw_img = Image.open(image_path).convert('RGB')
    img_tensor = infer_transform(raw_img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        confidence, predicted_idx = torch.max(probabilities, dim=0)

    predicted_class = CIFAR10_CLASSES[predicted_idx.item()]
    confidence_percent = confidence.item() * 100

    plt.imshow(raw_img)
    plt.title(f"Prediction: {predicted_class} ({confidence_percent:.2f}%)")
    plt.axis('off')
    plt.show()

    print(f"Class: {predicted_class} | Confidence: {confidence_percent:.2f}%")

sample_img, _ = test_dataset[0]
sample_img_pil = transforms.ToPILImage()(denormalize(sample_img).clip(0, 1))
sample_img_pil.save("test_sample.jpg")

predict_image("test_sample.jpg")

# 8. Comparative Insights & Deep Learning Analysis

### Comprehensive Technical Comparison: MNIST vs CIFAR-10

| Aspect | MNIST Dataset | CIFAR-10 Dataset |
| :--- | :--- | :--- |
| **Input Dimension** | `1 x 28 x 28` (1 Grayscale channel) | `3 x 32 x 32` (3 RGB color channels) |
| **Total Inputs per Sample** | $784$ floating-point values | $3,072$ floating-point values |
| **Visual Clarity to Humans** | Sharp high-contrast digits on black background | Highly pixelated and blurry when scaled ($32 \times 32$) |
| **Feature Complexity** | Structural strokes, curves, and loops | Complex textures, object contours, background clutter, and lighting variation |
| **Network Capacity Required** | 2 Conv Blocks, no Batch Normalization | 3 Conv Blocks + `BatchNorm2d` + `Dropout(0.5)` |
| **Data Augmentation Needed?** | No (Basic normalization is sufficient) | Yes (`RandomCrop`, `RandomHorizontalFlip` are vital) |
| **Regularization Necessity** | Low (Model rarely overfits) | High (Required to prevent overfitting on 50,000 train images) |
| **Training Epochs Needed** | **3 – 5 epochs** for **99%+** accuracy | **25 – 80 epochs** for **85%+ – 90%** accuracy |

### Core Analytical Insights
1. **Computer Vision Perception:** Human eyes struggle with low-resolution $32 \times 32$ CIFAR-10 images because we rely on high frequency details. CNNs decompose images into $3 \times 3$ kernel feature responses (edges, gradients, textures) which provide rich mathematical signal for accurate classification even at $32 \times 32$.
2. **The Necessity of BatchNorm & Dropout:** In CIFAR-10, deeper networks without `nn.BatchNorm2d` suffer from internal covariate shift and slow convergence. Without `nn.Dropout(0.5)` and `weight_decay`, models memorize training samples rather than learning generalizable patterns.
3. **Generalization via Augmentation:** Random spatial transformations force the network to become invariant to small translations and horizontal rotations, narrowing the gap between training and validation accuracy.